In [1]:
import warnings
warnings.filterwarnings('ignore')
import json
import dask.bag as db
import glob
import pandas as pd
from dask.distributed import Client
import re
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
#client = Client('10.233.24.232:8786')
from dask.delayed import delayed
pbar = ProgressBar()
pbar.register()
import numpy as np
import numpy.linalg as LA
import pandas as pd
import string
import itertools
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
#from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
#from wordcloud import WordCloud, ImageColorGenerator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from scipy.spatial.distance import euclidean
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn import metrics
from sklearn.preprocessing import scale
from sklearn.feature_extraction import text
%matplotlib inline

#wireless = db.read_text('https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Wireless_v1_00.tsv.gz')

KeyboardInterrupt: 

In [ ]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim import corpora, models

In [ ]:
filenames = ['amazon_reviews_us_Home_Improvement_v1_00.tsv.gz',
             'amazon_reviews_us_Lawn_and_Garden_v1_00.tsv.gz',
            'amazon_reviews_us_Kitchen_v1_00.tsv.gz',
           #'amazon_reviews_us_Home_Improvement_v1_00.tsv.gz'
# '/mnt/dat#a/public/amazon-reviews/amazon_reviews_us_Home_Entertainment_v1_00.tsv.gz',
# '/mnt/data/public/amazon-reviews/amazon_reviews_us_Home_v1_00.tsv.gz',
            'amazon_reviews_us_Health_Personal_Care_v1_00.tsv.gz']
# '/mnt/data/public/amazon-reviews/amazon_reviews_us_Grocery_v1_00.tsv.gz',
# '/mnt/data/public/amazon-reviews/amazon_reviews_us_Furniture_v1_00.tsv.gz',
# '/mnt/data/public/amazon-reviews/amazon_reviews_us_Apparel_v1_00.tsv.gz',
# '/mnt/data/public/amazon-reviews/amazon_reviews_us_Baby_v1_00.tsv.gz']

In [ ]:
import warnings
warnings.filterwarnings('ignore')
cols = ['marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date']
dfs = [delayed(pd.read_csv)(fn, delimiter='\t', error_bad_lines = False,  compression  ='gzip',
                            header = 0,
                           usecols = ['customer_id','product_category','star_rating'], dtype = {'star_rating':'object'}
                           )
       for fn in filenames]
# [delayed(pd.read_csv)(fn, delimiter='\t', error_bad_lines = False, skiprows = 1, compression  ='gzip')
#        for fn in filenames]
df = dd.from_delayed(dfs).persist()
#df.groupby('product_category').size().compute()

In [ ]:
#(?<=% of )(.*)(?= at )
import re
product_categories = []
for fn in filenames:
 #   fn = 'amazon_reviews_us_Lawn_and_Garden_v1_00.tsv.gz'
    line = re.search(r"""(?<=amazon_reviews_us_)(.*)(?=_v1_00.tsv.gz)""", fn)
    line =  line[1].replace("_"," ")
    product_categories.append(line)
product_categories.append('Health & Personal Care')

In [ ]:
dfPC = df[df.product_category.isin(product_categories)].compute()

In [ ]:
dfPC.head()

In [ ]:
dfCount = dfPC.groupby('product_category')['customer_id'].apply(lambda x: len(x.unique()))

In [ ]:
dfCount

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
plt.bar(dfCount.index, dfCount.values)
plt.xticks( rotation='25')
plt.xlabel("Product Categories")
plt.ylabel("Number of Customers")

In [ ]:
dfStarCount = dfPC.groupby(['product_category','star_rating'])['customer_id'].apply(lambda x: len(x.unique()))

In [ ]:
dfStarCount

In [ ]:
import seaborn as sns
markers = ['x','*','+','o','v','^','<','>','s','p']
#i = 0
ticks = np.arange(1,6,1)
#plt.
plt.figure(figsize=(15,7))
for i in range(0, int(dfStarCount.shape[0]/5)):
    plt.plot(ticks, dfStarCount.values[(i*5):((i+1)*5)], marker = markers[i], label = dfStarCount.index[i*5][0] )
    plt.xticks(ticks)
    plt.legend()
    plt.title("Star Ratings and Number of Customers")
    plt.xlabel("Star Ratings")
    plt.ylabel("Number of Customers")
### Change to Bar Chart

Comment: Bias because the category with high number of customers will receive high number of star ratings. 
<br>Solution: Normalize data.

In [ ]:
no_categories = int(dfStarCount.shape[0]/5)

fig, axes = plt.subplots(no_categories/2, 2, figsize = (15,12))

# fig, axes = plt.subplots(2,2, figsize=(15,12))
# axes[0,0].scatter(x=df.temp, y=df.cnt)
# axes[0,0].set_title("Temperature and Bike Rental Count")
# axes[0,1].scatter(x=df.atemp, y=df.cnt)
# axes[0,1].set_title("Actual Feel and Bike Rental Count")
# axes[1,0].scatter(x=df.hum, y=df.cnt)
# axes[1,0].set_title("Humidity and Bike Rental Count")
# axes[1,1].scatter(x=df.windspeed, y=df.cnt)
# axes[1,1].set_title("Windspeed and Bike Rental Count")

# import matplotlib.pyplot as plt
# import seaborn as sns; sns.set()
# plt.bar(dfCount.index, dfCount.values)
# plt.xticks( rotation='25')
# plt.xlabel("Product Categories")
# plt.ylabel("Number of Customers")

# import seaborn as sns
# markers = ['x','*','+','o','v','^','<','>','s','p']
# #i = 0
# ticks = np.arange(1,6,1)
# #plt.
# plt.figure(figsize=(15,7))
# for i in range(0, int(dfStarCount.shape[0]/5)):
#     plt.plot(ticks, dfStarCount.values[(i*5):((i+1)*5)], marker = markers[i], label = dfStarCount.index[i*5][0] )
#     plt.xticks(ticks)
#     plt.legend()
#     plt.title("Star Ratings and Number of Customers")
#     plt.xlabel("Star Ratings")
#     plt.ylabel("Number of Customers")
### Change to Bar Chart

In [ ]:
list_values = []
list_pc = []
for elem,i in zip(dfStarCount.index, range(1,dfStarCount.shape[0]+1)):
    
    if( i % 5==0):
        #print(dfStarCount.index[i-1][0])
        list_values.append(dfStarCount.values[(i-5):i])
        list_pc.append(dfStarCount.index[i-1][0])

norm_list = []
for i in range(0,int(dfStarCount.shape[0]/5)):
    raw = list_values[i]
    norm = [e/sum(raw) for e in raw]
    norm_list.append(norm)

In [ ]:
plt.figure(figsize=(15,7))
for i in range(0, int(dfStarCount.shape[0]/5)):
    plt.plot(ticks,norm_list[i], marker = markers[i], label = dfStarCount.index[i*5][0] )
    plt.xticks(ticks)
    plt.legend()
    plt.title("Star Ratings and Normalized Number of Customers")
    plt.xlabel("Star Ratings")
    plt.ylabel("Number of Customers")


In [ ]:
import warnings
warnings.filterwarnings('ignore')
cols = ['marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date']
dfs = [delayed(pd.read_csv)(fn, delimiter='\t', error_bad_lines = False,  compression  ='gzip',
                            header = 0,
                           usecols = ['customer_id','product_category','review_date'], dtype = {'review_date':'str'}
                           )
       for fn in filenames]
# [delayed(pd.read_csv)(fn, delimiter='\t', error_bad_lines = False, skiprows = 1, compression  ='gzip')
#        for fn in filenames]
dfYear = dd.from_delayed(dfs).persist()
#df.groupby('product_category').size().compute()

In [ ]:
dfYPC = dfYear[dfYear.product_category.isin(product_categories)]

In [ ]:
dfYPC.head()

In [ ]:
dfYPC['year'] = dfYPC['review_date'].str.slice(0,4)

In [ ]:
dfYPC.head()

In [ ]:
dfYPC['year'] = dfYPC['year'].astype(str)

In [ ]:
dfYPC['year'].unique().compute()

In [ ]:
list_years = np.arange(1999,2016,1)
list_years=list_years.astype(str)
dfYPC = dfYPC[dfYPC.year.isin(list_years)]

In [ ]:
dfYPC['year'] =dfYPC['year'].astype(int)

In [ ]:
dfYPC['year'].unique().compute()

In [ ]:
dfYDate = dfYPC.groupby(['product_category','year'])['customer_id'].apply(lambda x: len(x.unique())).persist()

In [ ]:
dfYDate.compute()

In [ ]:
dfYPC.compute()

In [ ]:
product_categories

In [ ]:
dfHM = dfYPC[dfYPC.product_category == product_categories[0]]

In [ ]:
dfHM = dfHM.drop(['review_date'], axis=1)

In [ ]:
dfHM.head()

In [ ]:
dfHM['year'] =dfHM['year'].astype(int)

In [ ]:
dfYCount = dfHM.groupby('year')['customer_id'].apply(lambda x: len(x.unique())).compute()

In [ ]:
dfYCount

In [ ]:
dfYCountSorted = dfYCount.sort_index()

In [ ]:
dfYCountSorted

In [ ]:
prod_categ = dfYPC.product_category.unique().compute()

In [ ]:
prod_categ.values

In [ ]:
#product_categories 
countCust = []
#dfHM = dfYPC[dfYPC.product_category == product_categories[0]]
for prod in prod_categ.values:
    newDF =  dfYPC[dfYPC.product_category == prod]
    dfYCount = newDF.groupby('year')['customer_id'].apply(lambda x: len(x.unique())).compute()
    dfYCountSorted = dfYCount.sort_index()
    countCust.append([dfYCountSorted.index,dfYCountSorted.values])
    

In [ ]:
#for elem in range (0,4):
print(countCust[0][0])

#countCust[0]

In [ ]:
prod_categ.values

In [ ]:
import seaborn as sns;sns.set()
markers = ['x','*','+','o','v','^','<','>','s','p']
#i = 0
ticks = dfYCountSorted.index
vals = dfYCountSorted.values
#plt.
plt.figure(figsize=(15,7))
for i,u in zip(prod_categ.values,range(0,len(prod_categ.values))):
    plt.plot(countCust[u][0], countCust[u][1], marker = markers[u], label = i )
    plt.xticks(ticks)
    plt.legend()
    plt.title("Years and Number of Customers")
    plt.xlabel("Years")
    plt.ylabel("Number of Customers")


In [ ]:
filenames = ['amazon_reviews_us_Home_Improvement_v1_00.tsv.gz',
             'amazon_reviews_us_Lawn_and_Garden_v1_00.tsv.gz',
            'amazon_reviews_us_Kitchen_v1_00.tsv.gz',
           #'amazon_reviews_us_Home_Improvement_v1_00.tsv.gz'
#             'amazon_reviews_us_Home_Entertainment_v1_00.tsv.gz',
#             'amazon_reviews_us_Home_v1_00.tsv.gz',
            'amazon_reviews_us_Health_Personal_Care_v1_00.tsv.gz']
#             'amazon_reviews_us_Grocery_v1_00.tsv.gz',
#             'amazon_reviews_us_Furniture_v1_00.tsv.gz',
#             'amazon_reviews_us_Apparel_v1_00.tsv.gz',
#             'amazon_reviews_us_Baby_v1_00.tsv.gz']

In [ ]:
import warnings
warnings.filterwarnings('ignore')
cols = ['marketplace', 'customer_id', 'review_id', 'product_id',
       'product_parent', 'product_title', 'product_category', 'star_rating',
       'helpful_votes', 'total_votes', 'vine', 'verified_purchase',
       'review_headline', 'review_body', 'review_date']
dfs = [delayed(pd.read_csv)(fn, delimiter='\t', error_bad_lines = False,  compression  ='gzip',
                            header = 0,
                           usecols = ['customer_id','product_category','verified_purchase','review_date'], dtype = {'review_date':'str'}
                           )
       for fn in filenames]
dfYear = dd.from_delayed(dfs).persist()